# Room 4.9 — Data Preprocessing and Integration

This notebook prepares the final dataset used for the analysis of Room 4.9.

The preprocessing pipeline combines three different sensor sources:

1. **Indoor Air Quality (IAQ) sensors**
   - CO₂ concentration
   - Temperature
   - Relative humidity
   - Light level
   - PIR motion detection

2. **Desk Occupancy sensors**
   - 32 individual desk occupancy sensors

3. **Magnetic Contact sensors**
   - 4 window-state sensors

All sensor measurements are cleaned, temporally aligned to common
10-minute intervals and integrated into a single dataset.

The final output of this notebook is:

`room_4_9_full_dataset.csv`


In [1]:
# IMPORTS AND PATH CONFIGURATION
# The notebook uses relative paths so that it can be executed
# both locally and from a cloned GitHub repository without
# depending on Google Colab-specific "/content/" paths.


from pathlib import Path
import json

import numpy as np
import pandas as pd


# Project directories

BASE_DIR = Path(".")

RAW_DATA_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"


# Create the processed-data directory if it does not exist.
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Raw data directory:")
print(RAW_DATA_DIR.resolve())

print("\nProcessed data directory:")
print(PROCESSED_DATA_DIR.resolve())

Raw data directory:
/content/data/raw

Processed data directory:
/content/data/processed


## 1. Indoor Air Quality (IAQ) preprocessing

Room 4.9 contains two IAQ sensors. Their raw telemetry is processed
independently and subsequently combined into a common 10-minute
representation of the environmental conditions inside the laboratory.

The preprocessing procedure includes:

- conversion of raw JSON telemetry to tabular long format,
- removal of known invalid sensor values,
- exclusion of previously identified anomalous periods,
- temporal aggregation into 10-minute intervals,
- limited interpolation of short gaps,
- integration of the two IAQ sensors,
- calculation of room-level environmental variables.


In [2]:
from pathlib import Path
import shutil

# Create project folders
RAW_DATA_DIR = Path("data/raw")
PROCESSED_DATA_DIR = Path("data/processed")

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Move/copy uploaded files from /content/ into data/raw/
source_files = [
    Path("/content/iaq_1_4_9_raw.json"),
    Path("/content/iaq_2_4_9_raw.json"),
]

for src in source_files:
    if src.exists():
        dst = RAW_DATA_DIR / src.name
        shutil.copy2(src, dst)
        print(f"Copied: {src.name}")
    else:
        print(f"Not found in /content/: {src.name}")

Copied: iaq_1_4_9_raw.json
Copied: iaq_2_4_9_raw.json


In [3]:
# LOAD RAW IAQ DATA


iaq1_file = RAW_DATA_DIR / "iaq_1_4_9_raw.json"
iaq2_file = RAW_DATA_DIR / "iaq_2_4_9_raw.json"


# Verify that both input files are available

print("IAQ-1 file exists:", iaq1_file.exists())
print("IAQ-2 file exists:", iaq2_file.exists())

if not iaq1_file.exists():
    raise FileNotFoundError(
        f"Missing input file: {iaq1_file}"
    )

if not iaq2_file.exists():
    raise FileNotFoundError(
        f"Missing input file: {iaq2_file}"
    )

# Load raw JSON telemetry

with open(
    iaq1_file,
    "r",
    encoding="utf-8"
) as file:
    iaq1_raw = json.load(file)


with open(
    iaq2_file,
    "r",
    encoding="utf-8"
) as file:
    iaq2_raw = json.load(file)


print("\nIAQ-1 telemetry variables:")
print(list(iaq1_raw.keys()))

print("\nIAQ-2 telemetry variables:")
print(list(iaq2_raw.keys()))


# Inspect the number of raw observations per variable

print("\nNumber of raw IAQ-1 observations:")

for key, values in iaq1_raw.items():
    print(
        f"{key}: {len(values)}"
    )


print("\nNumber of raw IAQ-2 observations:")

for key, values in iaq2_raw.items():
    print(
        f"{key}: {len(values)}"
    )

IAQ-1 file exists: True
IAQ-2 file exists: True

IAQ-1 telemetry variables:
['co2', 'temperature', 'humidity', 'light_level', 'pir']

IAQ-2 telemetry variables:
['co2', 'temperature', 'humidity', 'light_level', 'pir']

Number of raw IAQ-1 observations:
co2: 207540
temperature: 206542
humidity: 206542
light_level: 38991
pir: 38991

Number of raw IAQ-2 observations:
co2: 38033
temperature: 37210
humidity: 37210
light_level: 37210
pir: 37210


In [4]:
# CONVERT IAQ JSON TELEMETRY TO LONG FORMAT


def json_to_long(data, sensor_name):
    """
    Convert raw ThingsBoard-style JSON telemetry from one
    IAQ sensor into a long-format pandas DataFrame.

    Parameters
    ----------
    data : dict
        Raw JSON telemetry.
    sensor_name : str
        Identifier of the IAQ sensor.

    Returns
    -------
    pandas.DataFrame
        Long-format dataset containing timestamp, value,
        telemetry key and sensor identifier.
    """

    frames = []

    for key, records in data.items():

        # Skip empty or unexpected JSON entries.
        if (
            not isinstance(records, list)
            or len(records) == 0
        ):
            continue

        temp = pd.DataFrame(records)

        # Only telemetry entries containing both timestamp
        # and measurement value can be processed.
        if not {
            "ts",
            "value"
        }.issubset(temp.columns):
            continue

        temp["key"] = key
        temp["sensor"] = sensor_name

        frames.append(
            temp[
                [
                    "ts",
                    "value",
                    "key",
                    "sensor"
                ]
            ]
        )

    # Return an empty but correctly structured DataFrame
    # if no valid telemetry records are found.
    if not frames:

        return pd.DataFrame(
            columns=[
                "ts",
                "value",
                "key",
                "sensor"
            ]
        )

    df_long = pd.concat(
        frames,
        ignore_index=True
    )

    # Convert Unix timestamps from milliseconds to datetime.
    df_long["ts"] = pd.to_datetime(
        df_long["ts"],
        unit="ms",
        errors="coerce"
    )

    # Convert telemetry values to numeric format.
    df_long["value"] = pd.to_numeric(
        df_long["value"],
        errors="coerce"
    )

    # Remove invalid rows and duplicate observations.
    # If duplicate measurements exist for the same telemetry
    # variable and timestamp, the most recent occurrence is kept.
    df_long = (
        df_long
        .dropna(
            subset=[
                "ts",
                "value",
                "key"
            ]
        )
        .sort_values("ts")
        .drop_duplicates(
            subset=[
                "ts",
                "key"
            ],
            keep="last"
        )
        .reset_index(drop=True)
    )

    return df_long

In [5]:
# Convert the two IAQ sensors independently.

iaq1_long = json_to_long(
    iaq1_raw,
    sensor_name="IAQ-1"
)

iaq2_long = json_to_long(
    iaq2_raw,
    sensor_name="IAQ-2"
)


print(
    "IAQ-1 long-format shape:",
    iaq1_long.shape
)

print(
    "IAQ-2 long-format shape:",
    iaq2_long.shape
)


display(
    iaq1_long.head()
)

IAQ-1 long-format shape: (698606, 4)
IAQ-2 long-format shape: (186873, 4)


,ts,value,key,sensor
0,2025-08-31 21:01:57.989,0.0,pir,IAQ-1
1,2025-08-31 21:01:57.989,421.0,co2,IAQ-1
2,2025-08-31 21:01:57.989,29.3,temperature,IAQ-1
3,2025-08-31 21:01:57.989,43.0,humidity,IAQ-1
4,2025-08-31 21:01:57.989,0.0,light_level,IAQ-1


In [6]:
# REMOVE KNOWN INVALID IAQ READINGS


def remove_invalid_readings(df):
    """
    Remove measurements that correspond to known invalid
    sensor states or physically invalid values.

    The filtering rules reproduce the preprocessing used
    during the thesis analysis.
    """

    cleaned = df.copy()



    # Invalid CO₂ measurements
    # 65535 corresponds to a known sensor error value.
    # Non-positive CO₂ measurements are also considered invalid.

    invalid_co2 = (
        (cleaned["key"] == "co2")
        &
        (
            (cleaned["value"] == 65535)
            |
            (cleaned["value"] <= 0)
        )
    )


    # Invalid relative-humidity measurements


    invalid_humidity = (
        (cleaned["key"] == "humidity")
        &
        (
            (cleaned["value"] < 0)
            |
            (cleaned["value"] > 100)
        )
    )


    # Invalid PIR states
    # PIR is expected to be binary:
    # 0 = no detected motion
    # 1 = detected motion

    invalid_pir = (
        (cleaned["key"] == "pir")
        &
        (~cleaned["value"].isin([0, 1]))
    )


    invalid_mask = (
        invalid_co2
        |
        invalid_humidity
        |
        invalid_pir
    )


    print(
        "Invalid observations removed:",
        int(invalid_mask.sum())
    )


    cleaned = (
        cleaned
        .loc[~invalid_mask]
        .copy()
        .reset_index(drop=True)
    )

    return cleaned

In [7]:
iaq1_clean_long = remove_invalid_readings(
    iaq1_long
)

iaq2_clean_long = remove_invalid_readings(
    iaq2_long
)


print(
    "\nIAQ-1 observations after basic cleaning:",
    iaq1_clean_long.shape
)

print(
    "IAQ-2 observations after basic cleaning:",
    iaq2_clean_long.shape
)

Invalid observations removed: 1201
Invalid observations removed: 1007

IAQ-1 observations after basic cleaning: (697405, 4)
IAQ-2 observations after basic cleaning: (185866, 4)


In [8]:
# REMOVE PREVIOUSLY IDENTIFIED ANOMALOUS PERIODS — IAQ-1

# During the original data-quality assessment, two short
# periods were identified as problematic for the continuous
# IAQ variables of sensor IAQ-1.
#
# To reproduce the dataset used in the thesis analysis,
# measurements of CO₂, temperature and humidity within these
# intervals are excluded before temporal aggregation.


anomalous_periods = [
    (
        pd.Timestamp("2026-01-17 14:00:00"),
        pd.Timestamp("2026-01-17 16:00:00")
    ),
    (
        pd.Timestamp("2026-01-18 15:00:00"),
        pd.Timestamp("2026-01-18 17:00:00")
    )
]

continuous_keys = [
    "co2",
    "temperature",
    "humidity"
]

rows_before = len(iaq1_clean_long)

for start, end in anomalous_periods:

    anomaly_mask = (
        iaq1_clean_long["key"].isin(continuous_keys)
        &
        (iaq1_clean_long["ts"] >= start)
        &
        (iaq1_clean_long["ts"] < end)
    )

    iaq1_clean_long = (
        iaq1_clean_long
        .loc[~anomaly_mask]
        .copy()
    )

iaq1_clean_long = (
    iaq1_clean_long
    .sort_values("ts")
    .reset_index(drop=True)
)

removed_anomalous_rows = (
    rows_before - len(iaq1_clean_long)
)

print(
    "IAQ-1 observations removed from anomalous periods:",
    removed_anomalous_rows
)

print(
    "IAQ-1 shape after anomaly removal:",
    iaq1_clean_long.shape
)

IAQ-1 observations removed from anomalous periods: 276
IAQ-1 shape after anomaly removal: (697129, 4)


In [11]:
# TEMPORAL AGGREGATION TO 10-MINUTE INTERVALS
#
# Continuous environmental variables are aggregated using
# the median value within each 10-minute interval.
#
# PIR activity is aggregated using the maximum value so that
# any detected motion within an interval is preserved.
#
# Only short missing periods are filled:
# - continuous variables: time interpolation up to 30 minutes
# - discrete variables: forward filling up to 30 minutes


def resample_iaq(df):
    """
    Aggregate IAQ telemetry into common 10-minute intervals.

    Parameters
    ----------
    df : pandas.DataFrame
        Long-format IAQ telemetry containing timestamp,
        measurement value and telemetry key.

    Returns
    -------
    pandas.DataFrame
        Wide-format dataset aligned to 10-minute intervals.
    """

    aggregation = {
        "co2": "median",
        "temperature": "median",
        "humidity": "median",
        "light_level": "median",
        "pir": "max"
    }

    result = pd.DataFrame()

    for key, method in aggregation.items():

        subset = (
            df.loc[
                df["key"] == key,
                ["ts", "value"]
            ]
            .sort_values("ts")
            .set_index("ts")["value"]
        )

        if subset.empty:
            result[key] = pd.Series(dtype=float)
            continue

        series_10min = (
            subset
            .resample("10min")
            .agg(method)
        )

        if result.empty:
            result = series_10min.to_frame(
                name=key
            )

        else:
            result = result.join(
                series_10min.rename(key),
                how="outer"
            )

    result = result.sort_index()

    # Fill only short gaps in continuous variables.
    continuous_columns = [
        "co2",
        "temperature",
        "humidity"
    ]

    result[continuous_columns] = (
        result[continuous_columns]
        .interpolate(
            method="time",
            limit=3,
            limit_direction="both"
        )
    )

    # Discrete sensor states are not interpolated numerically.
    # The most recent known value is retained for up to
    # three consecutive 10-minute intervals.
    discrete_columns = [
        "light_level",
        "pir"
    ]

    result[discrete_columns] = (
        result[discrete_columns]
        .ffill(limit=3)
    )

    return result.reset_index()

In [12]:
iaq1_10min = resample_iaq(
    iaq1_clean_long
)

iaq2_10min = resample_iaq(
    iaq2_clean_long
)

print(
    "IAQ-1 10-minute shape:",
    iaq1_10min.shape
)

print(
    "IAQ-2 10-minute shape:",
    iaq2_10min.shape
)

print("\nIAQ-1 time range:")
print(
    iaq1_10min["ts"].min(),
    "→",
    iaq1_10min["ts"].max()
)

print("\nIAQ-2 time range:")
print(
    iaq2_10min["ts"].min(),
    "→",
    iaq2_10min["ts"].max()
)

IAQ-1 10-minute shape: (39948, 6)
IAQ-2 10-minute shape: (37069, 6)

IAQ-1 time range:
2025-08-31 21:00:00 → 2026-06-05 06:50:00

IAQ-2 time range:
2025-08-31 21:00:00 → 2026-05-16 07:00:00


In [13]:
# RENAME IAQ SENSOR VARIABLES

# Sensor-specific suffixes are added before merging the two
# IAQ datasets. This preserves the measurements from each
# sensor separately and avoids column-name conflicts.

iaq1_10min = iaq1_10min.rename(
    columns={
        "co2": "co2_iaq1",
        "temperature": "temperature_iaq1",
        "humidity": "humidity_iaq1",
        "light_level": "light_level_iaq1",
        "pir": "pir_iaq1"
    }
)

iaq2_10min = iaq2_10min.rename(
    columns={
        "co2": "co2_iaq2",
        "temperature": "temperature_iaq2",
        "humidity": "humidity_iaq2",
        "light_level": "light_level_iaq2",
        "pir": "pir_iaq2"
    }
)

print("IAQ sensor columns renamed successfully.")

IAQ sensor columns renamed successfully.


In [14]:
# INTEGRATE THE TWO IAQ SENSORS

# An outer merge is used to preserve timestamps for which
# measurements are available from only one of the two sensors.


df_iaq_4_9 = pd.merge(
    iaq1_10min,
    iaq2_10min,
    on="ts",
    how="outer"
)

df_iaq_4_9 = (
    df_iaq_4_9
    .sort_values("ts")
    .reset_index(drop=True)
)

print(
    "Shape after IAQ sensor integration:",
    df_iaq_4_9.shape
)

print("\nTime range:")
print(
    df_iaq_4_9["ts"].min(),
    "→",
    df_iaq_4_9["ts"].max()
)

display(df_iaq_4_9.head())

Shape after IAQ sensor integration: (39948, 11)

Time range:
2025-08-31 21:00:00 → 2026-06-05 06:50:00


,ts,co2_iaq1,temperature_iaq1,humidity_iaq1,light_level_iaq1,pir_iaq1,co2_iaq2,temperature_iaq2,humidity_iaq2,light_level_iaq2,pir_iaq2
0,2025-08-31 21:00:00,421.0,29.3,43.0,0.0,0.0,418.0,29.1,43.5,0.0,0.0
1,2025-08-31 21:10:00,421.0,29.3,43.0,0.0,0.0,418.0,29.1,43.5,0.0,0.0
2,2025-08-31 21:20:00,421.0,29.3,43.0,0.0,0.0,417.0,29.1,43.5,0.0,0.0
3,2025-08-31 21:30:00,421.0,29.3,43.0,0.0,0.0,417.0,29.1,43.5,0.0,0.0
4,2025-08-31 21:40:00,421.0,29.3,43.0,0.0,0.0,418.0,29.1,43.5,0.0,0.0


In [15]:
# CREATE ROOM-LEVEL ENVIRONMENTAL VARIABLES

# When measurements from both IAQ sensors are available,
# their arithmetic mean is used to obtain a single room-level
# representation of the environmental conditions.
#
# If only one IAQ sensor is available at a timestamp,
# pandas mean(..., skipna=True) retains the available value.
#
# PIR activity is combined using the maximum state so that
# motion detected by either sensor is preserved.


df_iaq_4_9["co2_avg"] = (
    df_iaq_4_9[
        ["co2_iaq1", "co2_iaq2"]
    ]
    .mean(
        axis=1,
        skipna=True
    )
)

df_iaq_4_9["temperature_avg"] = (
    df_iaq_4_9[
        [
            "temperature_iaq1",
            "temperature_iaq2"
        ]
    ]
    .mean(
        axis=1,
        skipna=True
    )
)

df_iaq_4_9["humidity_avg"] = (
    df_iaq_4_9[
        [
            "humidity_iaq1",
            "humidity_iaq2"
        ]
    ]
    .mean(
        axis=1,
        skipna=True
    )
)

df_iaq_4_9["light_level_avg"] = (
    df_iaq_4_9[
        [
            "light_level_iaq1",
            "light_level_iaq2"
        ]
    ]
    .mean(
        axis=1,
        skipna=True
    )
)

df_iaq_4_9["pir_any"] = (
    df_iaq_4_9[
        [
            "pir_iaq1",
            "pir_iaq2"
        ]
    ]
    .max(
        axis=1,
        skipna=True
    )
)

# Number of IAQ sensors contributing a valid CO₂ value
# at each timestamp.
df_iaq_4_9["available_iaq_sensors"] = (
    df_iaq_4_9[
        [
            "co2_iaq1",
            "co2_iaq2"
        ]
    ]
    .notna()
    .sum(axis=1)
)

display(
    df_iaq_4_9[
        [
            "ts",
            "co2_avg",
            "temperature_avg",
            "humidity_avg",
            "light_level_avg",
            "pir_any",
            "available_iaq_sensors"
        ]
    ].head()
)

,ts,co2_avg,temperature_avg,humidity_avg,light_level_avg,pir_any,available_iaq_sensors
0,2025-08-31 21:00:00,419.5,29.2,43.25,0.0,0.0,2
1,2025-08-31 21:10:00,419.5,29.2,43.25,0.0,0.0,2
2,2025-08-31 21:20:00,419.0,29.2,43.25,0.0,0.0,2
3,2025-08-31 21:30:00,419.0,29.2,43.25,0.0,0.0,2
4,2025-08-31 21:40:00,419.5,29.2,43.25,0.0,0.0,2


In [16]:
# FINAL IAQ DATASET VALIDATION

# Only timestamps containing all room-level variables required
# by the subsequent analyses are retained.


required_columns = [
    "co2_avg",
    "temperature_avg",
    "humidity_avg",
    "light_level_avg",
    "pir_any"
]

df_iaq_4_9_final = (
    df_iaq_4_9
    .dropna(
        subset=required_columns
    )
    .copy()
    .reset_index(drop=True)
)


# Ensure binary PIR representation.
df_iaq_4_9_final["pir_any"] = (
    df_iaq_4_9_final["pir_any"]
    .round()
    .clip(0, 1)
    .astype(int)
)


# Number of available IAQ sensors is a discrete variable.
df_iaq_4_9_final["available_iaq_sensors"] = (
    df_iaq_4_9_final[
        "available_iaq_sensors"
    ]
    .astype(int)
)


print(
    "Final IAQ dataset shape:",
    df_iaq_4_9_final.shape
)

print("\nTime range:")
print(
    df_iaq_4_9_final["ts"].min(),
    "→",
    df_iaq_4_9_final["ts"].max()
)

print("\nDuplicate timestamps:")
print(
    df_iaq_4_9_final[
        "ts"
    ].duplicated().sum()
)

print("\nMissing values:")
print(
    df_iaq_4_9_final[
        required_columns
    ].isna().sum()
)

print("\nAvailable IAQ sensors:")
print(
    df_iaq_4_9_final[
        "available_iaq_sensors"
    ]
    .value_counts()
    .sort_index()
)

print("\nCO₂ average statistics:")
print(
    df_iaq_4_9_final[
        "co2_avg"
    ].describe()
)

display(
    df_iaq_4_9_final.head()
)

Final IAQ dataset shape: (38834, 17)

Time range:
2025-08-31 21:00:00 → 2026-06-05 06:50:00

Duplicate timestamps:
0

Missing values:
co2_avg            0
temperature_avg    0
humidity_avg       0
light_level_avg    0
pir_any            0
dtype: int64

Available IAQ sensors:
available_iaq_sensors
1     1777
2    37057
Name: count, dtype: int64

CO₂ average statistics:
count    38834.000000
mean       518.549412
std        218.768506
min        368.000000
25%        418.500000
50%        452.000000
75%        517.500000
max       3889.000000
Name: co2_avg, dtype: float64


,ts,co2_iaq1,temperature_iaq1,humidity_iaq1,light_level_iaq1,pir_iaq1,co2_iaq2,temperature_iaq2,humidity_iaq2,light_level_iaq2,pir_iaq2,co2_avg,temperature_avg,humidity_avg,light_level_avg,pir_any,available_iaq_sensors
0,2025-08-31 21:00:00,421.0,29.3,43.0,0.0,0.0,418.0,29.1,43.5,0.0,0.0,419.5,29.2,43.25,0.0,0,2
1,2025-08-31 21:10:00,421.0,29.3,43.0,0.0,0.0,418.0,29.1,43.5,0.0,0.0,419.5,29.2,43.25,0.0,0,2
2,2025-08-31 21:20:00,421.0,29.3,43.0,0.0,0.0,417.0,29.1,43.5,0.0,0.0,419.0,29.2,43.25,0.0,0,2
3,2025-08-31 21:30:00,421.0,29.3,43.0,0.0,0.0,417.0,29.1,43.5,0.0,0.0,419.0,29.2,43.25,0.0,0,2
4,2025-08-31 21:40:00,421.0,29.3,43.0,0.0,0.0,418.0,29.1,43.5,0.0,0.0,419.5,29.2,43.25,0.0,0,2


In [17]:
# EXPORT CLEAN IAQ DATASET


iaq_output_file = (
    PROCESSED_DATA_DIR
    / "iaq_4_9_clean.csv"
)

df_iaq_4_9_final.to_csv(
    iaq_output_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Saved: {iaq_output_file}"
)

Saved: data/processed/iaq_4_9_clean.csv


## 2. Desk Occupancy Preprocessing

Room 4.9 contains 32 individual Desk Occupancy sensors, corresponding
to the 32 available workstations in the laboratory.

The raw occupancy measurements are processed independently for each
desk and subsequently aligned to a common 10-minute temporal resolution.

The preprocessing procedure includes:

- loading the 32 raw Desk Occupancy JSON files,
- conversion of timestamps and occupancy states,
- transformation from long to wide format,
- temporal alignment to 10-minute intervals,
- restriction to the common observation period,
- calculation of aggregate room-level occupancy indicators.

The final output of this section is:

`desk_occupancy_4_9_clean.csv`

In [28]:
# DESK OCCUPANCY — REQUIRED IMPORTS


import json
import os

import numpy as np
import pandas as pd

print("Desk Occupancy libraries loaded successfully.")

Desk Occupancy libraries loaded successfully.


In [29]:
# LOCATE RAW DESK OCCUPANCY FILES


raw_files = {}

for i in range(1, 33):

    filename = (
        f"/content/F4_4.9-Desk-{i}_raw.json"
    )

    if os.path.exists(filename):
        raw_files[i] = filename


print(
    f"Found {len(raw_files)} of 32 "
    "Desk Occupancy files."
)


# Report any missing files.
if len(raw_files) != 32:

    missing = [
        f"F4_4.9-Desk-{i}_raw.json"
        for i in range(1, 33)
        if i not in raw_files
    ]

    print("\nMissing files:")

    for filename in missing:
        print(filename)

Found 32 of 32 Desk Occupancy files.


In [30]:
# LOAD RAW DESK OCCUPANCY DATA

# Each JSON file corresponds to one desk sensor.
# All observations are combined into a single long-format
# DataFrame containing timestamp, occupancy state and desk ID.

all_dfs = []

for desk_id, filename in raw_files.items():

    with open(
        filename,
        "r",
        encoding="utf-8"
    ) as file:

        raw = json.load(file)

    # Extract occupancy telemetry.
    temp_df = pd.DataFrame(
        raw["occupancy"]
    )

    # Convert Unix timestamps from milliseconds to datetime.
    temp_df["ts"] = pd.to_datetime(
        temp_df["ts"],
        unit="ms",
        errors="coerce"
    )

    # Convert occupancy measurements to numeric values.
    temp_df["value"] = pd.to_numeric(
        temp_df["value"],
        errors="coerce"
    )

    # Store the corresponding desk identifier.
    temp_df["desk"] = desk_id

    all_dfs.append(temp_df)


# Combine measurements from all 32 desk sensors.
df_long = pd.concat(
    all_dfs,
    ignore_index=True
)


print(
    "Total raw Desk Occupancy observations:",
    len(df_long)
)

print(
    "Long-format shape:",
    df_long.shape
)

display(
    df_long.head()
)

Total raw Desk Occupancy observations: 34452
Long-format shape: (34452, 3)


,ts,value,desk
0,2026-07-05 15:27:06.739,0,1
1,2026-07-04 15:27:09.048,0,1
2,2026-07-03 15:27:11.365,0,1
3,2026-07-02 15:27:13.695,0,1
4,2026-07-01 15:27:15.996,0,1


In [31]:
# INITIAL DATA QUALITY ASSESSMENT

print("Missing values:")
print(
    df_long.isna().sum()
)

print("\nUnique occupancy values:")
print(
    sorted(
        df_long[
            "value"
        ]
        .dropna()
        .unique()
    )
)

print(
    "\nDuplicate observations "
    "(desk + timestamp):"
)

print(
    df_long
    .duplicated(
        subset=[
            "desk",
            "ts"
        ]
    )
    .sum()
)

print("\nRaw time range:")

print(
    df_long["ts"].min(),
    "→",
    df_long["ts"].max()
)

Missing values:
ts       0
value    0
desk     0
dtype: int64

Unique occupancy values:
[np.int64(0), np.int64(1)]

Duplicate observations (desk + timestamp):
0

Raw time range:
2025-08-31 21:16:27.887000 → 2026-07-05 20:34:08.518000


In [32]:
# SORT DESK OCCUPANCY OBSERVATIONS


df_long = (
    df_long
    .sort_values(
        [
            "desk",
            "ts"
        ]
    )
    .reset_index(drop=True)
)


# Occupancy is a binary state (0 or 1).
df_long["value"] = (
    df_long["value"]
    .astype(int)
)


display(
    df_long.head()
)

,ts,value,desk
0,2025-09-01 15:39:06.993,0,1
1,2025-09-02 15:39:04.695,0,1
2,2025-09-03 15:39:02.394,0,1
3,2025-09-04 15:39:00.097,0,1
4,2025-09-05 15:38:57.793,0,1


In [33]:
# CREATE 10-MINUTE DESK OCCUPANCY DATASET

# Measurements are assigned to 10-minute intervals.
# For each desk and interval, the latest recorded occupancy
# state is retained.
#
# The dataset is then transformed from long format into
# wide format, with one column for each desk.


df_long["ts_10min"] = (
    df_long["ts"]
    .dt.floor("10min")
)


df_desks = (
    df_long
    .pivot_table(
        index="ts_10min",
        columns="desk",
        values="value",
        aggfunc="last"
    )
    .sort_index()
)


# Create a regular 10-minute time series and propagate
# the most recently observed occupancy state.
df_desks = (
    df_desks
    .resample("10min")
    .ffill()
)


# Rename desk columns consistently.
df_desks.columns = [
    f"desk_{int(column)}"
    for column in df_desks.columns
]


# Restore timestamp as a regular column.
df_desks = (
    df_desks
    .rename_axis("ts")
    .reset_index()
)

df_desks.columns.name = None


print(
    "10-minute Desk Occupancy shape:",
    df_desks.shape
)

display(
    df_desks.head()
)

10-minute Desk Occupancy shape: (44349, 33)


,ts,desk_1,desk_2,desk_3,desk_4,desk_5,desk_6,desk_7,desk_8,desk_9,...,desk_23,desk_24,desk_25,desk_26,desk_27,desk_28,desk_29,desk_30,desk_31,desk_32
0,2025-08-31 21:10:00,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-08-31 21:20:00,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-08-31 21:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-08-31 21:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,2025-08-31 21:50:00,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
# REMOVE INITIAL PERIOD WITH INCOMPLETE DESK COVERAGE

# Desk sensors did not necessarily begin recording at exactly
# the same timestamp. The initial period is therefore removed
# until all desk columns contain an available state.


first_valid = (
    df_desks
    .drop(columns=["ts"])
    .apply(
        lambda col:
        col.first_valid_index()
    )
)

start_index = (
    first_valid.max()
)


df_desks = (
    df_desks
    .iloc[start_index:]
    .reset_index(drop=True)
)


print(
    "Shape after initial coverage restriction:",
    df_desks.shape
)

print("\nTime range:")

print(
    df_desks["ts"].min(),
    "→",
    df_desks["ts"].max()
)

print("\nMissing values:")

print(
    df_desks.isna().sum()
)

Shape after initial coverage restriction: (44208, 33)

Time range:
2025-09-01 20:40:00 → 2026-07-05 20:30:00

Missing values:
ts             0
desk_1     41625
desk_2     42832
desk_3     42652
desk_4     41336
desk_5     42441
desk_6     42485
desk_7     43014
desk_8     42804
desk_9     40960
desk_10    42150
desk_11    40383
desk_12    42232
desk_13    42627
desk_14    41415
desk_15    40957
desk_16    43280
desk_17    43289
desk_18    41399
desk_19    40418
desk_20    42605
desk_21    42446
desk_22    42324
desk_23    42201
desk_24    43401
desk_25    43175
desk_26    41546
desk_27    42393
desk_28    41983
desk_29    42669
desk_30    42338
desk_31    42267
desk_32    42875
dtype: int64


In [35]:
# DETERMINE COMMON OBSERVATION PERIOD

# The final Desk Occupancy dataset is restricted to the
# temporal interval for which all 32 sensors have measurement
# coverage.


coverage = (
    df_long
    .groupby("desk")["ts"]
    .agg(
        first_measurement="min",
        last_measurement="max"
    )
)


common_start = (
    coverage[
        "first_measurement"
    ]
    .max()
    .floor("10min")
)


common_end = (
    coverage[
        "last_measurement"
    ]
    .min()
    .floor("10min")
)


print(
    "Common start:",
    common_start
)

print(
    "Common end:",
    common_end
)


display(
    coverage
)

Common start: 2025-09-01 20:40:00
Common end: 2026-07-04 21:00:00


,first_measurement,last_measurement
desk,,
1,2025-09-01 15:39:06.993,2026-07-05 15:27:06.739
2,2025-08-31 21:16:27.887,2026-07-04 21:06:33.844
3,2025-09-01 09:44:53.936,2026-07-05 15:02:11.324
4,2025-09-01 08:48:04.608,2026-07-05 19:03:14.269
5,2025-08-31 21:56:37.392,2026-07-04 21:44:06.114
6,2025-09-01 14:33:08.619,2026-07-05 14:21:47.555
7,2025-09-01 07:38:43.971,2026-07-05 07:25:58.614
8,2025-09-01 20:13:14.424,2026-07-05 20:03:05.217
9,2025-09-01 11:05:11.464,2026-07-05 10:54:44.590


In [36]:
# RESTRICT DATASET TO COMMON SENSOR COVERAGE


df_desks = (
    df_desks[
        (df_desks["ts"] >= common_start)
        &
        (df_desks["ts"] <= common_end)
    ]
    .reset_index(drop=True)
)


print(
    "Shape after common-period restriction:",
    df_desks.shape
)


print("\nTime range:")

print(
    df_desks["ts"].min(),
    "→",
    df_desks["ts"].max()
)


display(
    df_desks.head()
)

display(
    df_desks.tail()
)

Shape after common-period restriction: (44067, 33)

Time range:
2025-09-01 20:40:00 → 2026-07-04 21:00:00


,ts,desk_1,desk_2,desk_3,desk_4,desk_5,desk_6,desk_7,desk_8,desk_9,...,desk_23,desk_24,desk_25,desk_26,desk_27,desk_28,desk_29,desk_30,desk_31,desk_32
0,2025-09-01 20:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-09-01 20:50:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-09-01 21:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-09-01 21:10:00,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-09-01 21:20:00,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,ts,desk_1,desk_2,desk_3,desk_4,desk_5,desk_6,desk_7,desk_8,desk_9,...,desk_23,desk_24,desk_25,desk_26,desk_27,desk_28,desk_29,desk_30,desk_31,desk_32
44062,2026-07-04 20:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44063,2026-07-04 20:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44064,2026-07-04 20:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44065,2026-07-04 20:50:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44066,2026-07-04 21:00:00,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# VALIDATE ALIGNED DESK OCCUPANCY DATA


print("Missing values:")

print(
    df_desks.isna().sum()
)


print("\nDuplicate timestamps:")

print(
    df_desks[
        "ts"
    ]
    .duplicated()
    .sum()
)


print("\nUnique values per desk:")

for column in df_desks.columns[1:]:

    values = sorted(
        df_desks[
            column
        ]
        .dropna()
        .unique()
    )

    print(
        f"{column}: {values}"
    )

Missing values:
ts             0
desk_1     41498
desk_2     42692
desk_3     42513
desk_4     41201
desk_5     42301
desk_6     42348
desk_7     42874
desk_8     42664
desk_9     40829
desk_10    42012
desk_11    40255
desk_12    42093
desk_13    42489
desk_14    41282
desk_15    40830
desk_16    43141
desk_17    43150
desk_18    41267
desk_19    40288
desk_20    42465
desk_21    42306
desk_22    42187
desk_23    42064
desk_24    43261
desk_25    43036
desk_26    41411
desk_27    42257
desk_28    41849
desk_29    42530
desk_30    42210
desk_31    42136
desk_32    42735
dtype: int64

Duplicate timestamps:
0

Unique values per desk:
desk_1: [np.float64(0.0), np.float64(1.0)]
desk_2: [np.float64(0.0), np.float64(1.0)]
desk_3: [np.float64(0.0), np.float64(1.0)]
desk_4: [np.float64(0.0), np.float64(1.0)]
desk_5: [np.float64(0.0), np.float64(1.0)]
desk_6: [np.float64(0.0), np.float64(1.0)]
desk_7: [np.float64(0.0), np.float64(1.0)]
desk_8: [np.float64(0.0), np.float64(1.0)]
desk_9: [np.floa

In [42]:
# COMPLETE DESK OCCUPANCY STATES

# Desk occupancy sensors report state changes rather than
# necessarily providing a new observation at every 10-minute
# interval.
#
# Therefore, the most recently observed state of each desk is
# propagated forward until a new state is recorded.
#
# Remaining missing values, occurring before the first known
# state of a sensor, are treated as unoccupied (0).


desk_columns = [
    f"desk_{i}"
    for i in range(1, 33)
]

df_desks[desk_columns] = (
    df_desks[desk_columns]
    .ffill()
    .fillna(0)
)

# Convert occupancy states to integers:
# 0 = unoccupied
# 1 = occupied
df_desks[desk_columns] = (
    df_desks[desk_columns]
    .astype(int)
)

print("Missing values after state propagation:")
print(
    df_desks[desk_columns]
    .isna()
    .sum()
)

display(df_desks.head())

Missing values after state propagation:
desk_1     0
desk_2     0
desk_3     0
desk_4     0
desk_5     0
desk_6     0
desk_7     0
desk_8     0
desk_9     0
desk_10    0
desk_11    0
desk_12    0
desk_13    0
desk_14    0
desk_15    0
desk_16    0
desk_17    0
desk_18    0
desk_19    0
desk_20    0
desk_21    0
desk_22    0
desk_23    0
desk_24    0
desk_25    0
desk_26    0
desk_27    0
desk_28    0
desk_29    0
desk_30    0
desk_31    0
desk_32    0
dtype: int64


,ts,desk_1,desk_2,desk_3,desk_4,desk_5,desk_6,desk_7,desk_8,desk_9,...,desk_27,desk_28,desk_29,desk_30,desk_31,desk_32,occupied_desks,free_desks,occupancy_rate,any_desk_occupied
0,2025-09-01 20:40:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.0,32.0,0.0,0
1,2025-09-01 20:50:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.0,32.0,0.0,0
2,2025-09-01 21:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.0,32.0,0.0,0
3,2025-09-01 21:10:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.0,32.0,0.0,0
4,2025-09-01 21:20:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.0,32.0,0.0,0


In [43]:
# CALCULATE ROOM-LEVEL OCCUPANCY VARIABLES


desk_columns = [
    f"desk_{i}"
    for i in range(1, 33)
]


# Number of occupied desks at each timestamp.
df_desks["occupied_desks"] = (
    df_desks[
        desk_columns
    ]
    .sum(axis=1)
)


# Number of available/free desks.
df_desks["free_desks"] = (
    32
    - df_desks[
        "occupied_desks"
    ]
)


# Proportion of occupied desks.
df_desks["occupancy_rate"] = (
    df_desks[
        "occupied_desks"
    ]
    / 32
)


# Binary room-level occupancy indicator.
df_desks["any_desk_occupied"] = (
    df_desks[
        "occupied_desks"
    ]
    > 0
).astype(int)


print(
    "Dataset shape:",
    df_desks.shape
)


display(
    df_desks[
        [
            "ts",
            "occupied_desks",
            "free_desks",
            "occupancy_rate",
            "any_desk_occupied"
        ]
    ].head()
)

Dataset shape: (44067, 37)


,ts,occupied_desks,free_desks,occupancy_rate,any_desk_occupied
0,2025-09-01 20:40:00,0,32,0.0,0
1,2025-09-01 20:50:00,0,32,0.0,0
2,2025-09-01 21:00:00,0,32,0.0,0
3,2025-09-01 21:10:00,0,32,0.0,0
4,2025-09-01 21:20:00,0,32,0.0,0


In [44]:
# FINAL DESK OCCUPANCY VALIDATION

print(
    "Final Desk Occupancy shape:",
    df_desks.shape
)


print("\nTime range:")

print(
    df_desks["ts"].min(),
    "→",
    df_desks["ts"].max()
)


print("\nMissing values:")

print(
    df_desks.isna().sum()
)


print("\nDuplicate timestamps:")

print(
    df_desks[
        "ts"
    ]
    .duplicated()
    .sum()
)


print("\nOccupied desks statistics:")

print(
    df_desks[
        "occupied_desks"
    ]
    .describe()
)


print("\nOccupancy rate statistics:")

print(
    df_desks[
        "occupancy_rate"
    ]
    .describe()
)


print("\nAny desk occupied:")

print(
    df_desks[
        "any_desk_occupied"
    ]
    .value_counts()
    .sort_index()
)

Final Desk Occupancy shape: (44067, 37)

Time range:
2025-09-01 20:40:00 → 2026-07-04 21:00:00

Missing values:
ts                   0
desk_1               0
desk_2               0
desk_3               0
desk_4               0
desk_5               0
desk_6               0
desk_7               0
desk_8               0
desk_9               0
desk_10              0
desk_11              0
desk_12              0
desk_13              0
desk_14              0
desk_15              0
desk_16              0
desk_17              0
desk_18              0
desk_19              0
desk_20              0
desk_21              0
desk_22              0
desk_23              0
desk_24              0
desk_25              0
desk_26              0
desk_27              0
desk_28              0
desk_29              0
desk_30              0
desk_31              0
desk_32              0
occupied_desks       0
free_desks           0
occupancy_rate       0
any_desk_occupied    0
dtype: int64

Duplicate timestamps:
0

In [45]:
# SAVE CLEAN DESK OCCUPANCY DATASET


output_file = (
    "desk_occupancy_4_9_clean.csv"
)


df_desks.to_csv(
    output_file,
    index=False
)


print(
    f"Saved: {output_file}"
)

print(
    "Final shape:",
    df_desks.shape
)

Saved: desk_occupancy_4_9_clean.csv
Final shape: (44067, 37)


## 3. Magnetic Contact Sensor Preprocessing

Room 4.9 contains four magnetic contact sensors used to monitor the state of the laboratory windows. These sensors provide binary measurements indicating the open or closed state of each monitored window.

The preprocessing procedure includes:

- loading the raw JSON telemetry from the four magnetic contact sensors,
- converting timestamps and sensor values into appropriate formats,
- combining the measurements into a common dataset,
- aligning the sensor states to 10-minute intervals,
- propagating the latest known window state between consecutive observations,
- calculating the number of open windows,
- creating a binary indicator representing whether at least one window is open.

The resulting dataset is exported as:

`magnetic_contacts_4_9_clean.csv`

In [46]:
# 3.1 Define input files


mc_files = {
    1: "/content/F4_4.9-MC-1_raw.json",
    2: "/content/F4_4.9-MC-2_raw.json",
    3: "/content/F4_4.9-MC-3_raw.json",
    4: "/content/F4_4.9-MC-4_raw.json",
}


# Check whether all expected files are available.

missing_files = [
    path
    for path in mc_files.values()
    if not os.path.exists(path)
]

print(
    f"Found {4 - len(missing_files)} "
    f"of 4 magnetic contact files."
)

if missing_files:
    print("\nMissing files:")

    for path in missing_files:
        print(path)

Found 4 of 4 magnetic contact files.


In [47]:
# 3.2 Load and combine raw magnetic-contact telemetry


all_dfs = []

for sensor_id, path in mc_files.items():

    # Load the raw JSON file.
    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        raw = json.load(file)

    # Extract magnetic-contact measurements.
    records = raw.get(
        "magnet_status",
        []
    )

    df_sensor = pd.DataFrame(records)

    # Convert Unix timestamps from milliseconds to datetime.
    df_sensor["ts"] = pd.to_datetime(
        df_sensor["ts"],
        unit="ms",
        errors="coerce"
    )

    # Convert sensor states to numeric values.
    df_sensor["value"] = pd.to_numeric(
        df_sensor["value"],
        errors="coerce"
    )

    # Add sensor identifier.
    df_sensor["sensor"] = (
        f"mc_{sensor_id}"
    )

    all_dfs.append(df_sensor)


# Combine all four sensors into a single long-format dataset.

df_mc_long = pd.concat(
    all_dfs,
    ignore_index=True
)


print(
    "Total magnetic-contact observations:",
    len(df_mc_long)
)

display(
    df_mc_long.head()
)

Total magnetic-contact observations: 10121


,ts,value,sensor
0,2026-07-05 19:58:09.070,0,mc_1
1,2026-07-05 19:57:56.947,1,mc_1
2,2026-07-05 13:46:39.633,0,mc_1
3,2026-07-04 21:08:23.493,0,mc_1
4,2026-07-04 21:08:06.304,1,mc_1


In [49]:
# 3.3 Initial data-quality assessment

print(
    "Long-format shape:",
    df_mc_long.shape
)

print("\nMissing values:")
print(
    df_mc_long.isna().sum()
)

print("\nDuplicate sensor-timestamp observations:")
print(
    df_mc_long
    .duplicated(
        subset=[
            "sensor",
            "ts"
        ]
    )
    .sum()
)

print("\nUnique sensor states:")
print(
    sorted(
        df_mc_long["value"].unique()
    )
)

print("\nTime range:")

print(
    df_mc_long["ts"].min(),
    "→",
    df_mc_long["ts"].max()
)

Long-format shape: (10121, 3)

Missing values:
ts        0
value     0
sensor    0
dtype: int64

Duplicate sensor-timestamp observations:
0

Unique sensor states:
[np.int64(0), np.int64(1)]

Time range:
2025-09-01 08:33:06.181000 → 2026-07-05 19:58:09.070000


In [50]:
# 3.4 Temporal alignment to 10-minute intervals


# Magnetic-contact measurements represent discrete window
# states. Each observation is assigned to a 10-minute interval.
#
# The most recent state within each interval is retained.
# Forward filling is then used because the latest observed
# magnetic-contact state remains valid until a new state is
# reported.



# Assign each observation to a 10-minute interval.

df_mc_long["ts_10min"] = (
    df_mc_long["ts"]
    .dt.floor("10min")
)


# Transform the dataset from long to wide format.
# Each magnetic-contact sensor becomes a separate column.

df_mc = (
    df_mc_long
    .pivot_table(
        index="ts_10min",
        columns="sensor",
        values="value",
        aggfunc="last"
    )
    .sort_index()
)


# Create a continuous 10-minute time series and propagate
# the latest known state of each magnetic contact.

df_mc = (
    df_mc
    .resample("10min")
    .ffill()
)


# Restore timestamp as a standard DataFrame column.

df_mc = (
    df_mc
    .rename_axis("ts")
    .reset_index()
)

df_mc.columns.name = None


display(
    df_mc.head()
)

print(
    "10-minute dataset shape:",
    df_mc.shape
)

,ts,mc_1,mc_2,mc_3,mc_4
0,2025-09-01 08:30:00,1.0,1.0,NaN,NaN
1,2025-09-01 08:40:00,1.0,1.0,NaN,NaN
2,2025-09-01 08:50:00,1.0,1.0,NaN,NaN
3,2025-09-01 09:00:00,0.0,NaN,NaN,NaN
4,2025-09-01 09:10:00,0.0,NaN,NaN,NaN


10-minute dataset shape: (44277, 5)


In [51]:
# 3.5 Ensure availability of all four sensors


expected_sensors = [
    "mc_1",
    "mc_2",
    "mc_3",
    "mc_4"
]


# Create any missing sensor column if necessary.

for column in expected_sensors:

    if column not in df_mc.columns:
        df_mc[column] = np.nan


# Propagate the latest available state.
# Remaining values before the first recorded state are
# initialized to zero.

df_mc[expected_sensors] = (
    df_mc[expected_sensors]
    .ffill()
    .fillna(0)
)


display(
    df_mc.head()
)

,ts,mc_1,mc_2,mc_3,mc_4
0,2025-09-01 08:30:00,1.0,1.0,0.0,0.0
1,2025-09-01 08:40:00,1.0,1.0,0.0,0.0
2,2025-09-01 08:50:00,1.0,1.0,0.0,0.0
3,2025-09-01 09:00:00,0.0,1.0,0.0,0.0
4,2025-09-01 09:10:00,0.0,1.0,0.0,0.0


In [52]:
# 3.6 Create room-level window-state variables



# Number of open windows at each 10-minute interval.

df_mc["open_windows"] = (
    df_mc[
        [
            "mc_1",
            "mc_2",
            "mc_3",
            "mc_4"
        ]
    ]
    .sum(axis=1)
)


# Binary indicator:
#   0 = no window is open
#   1 = at least one window is open

df_mc["open_windows_binary"] = (
    df_mc["open_windows"] > 0
).astype(int)


print(
    "Dataset shape:",
    df_mc.shape
)

display(
    df_mc.head()
)

Dataset shape: (44277, 7)


,ts,mc_1,mc_2,mc_3,mc_4,open_windows,open_windows_binary
0,2025-09-01 08:30:00,1.0,1.0,0.0,0.0,2.0,1
1,2025-09-01 08:40:00,1.0,1.0,0.0,0.0,2.0,1
2,2025-09-01 08:50:00,1.0,1.0,0.0,0.0,2.0,1
3,2025-09-01 09:00:00,0.0,1.0,0.0,0.0,1.0,1
4,2025-09-01 09:10:00,0.0,1.0,0.0,0.0,1.0,1


In [53]:
# 3.7 Final validation and export


print(
    "Final Magnetic Contact shape:",
    df_mc.shape
)

print("\nTime range:")

print(
    df_mc["ts"].min(),
    "→",
    df_mc["ts"].max()
)

print("\nMissing values:")

print(
    df_mc.isna().sum()
)

print("\nDuplicate timestamps:")

print(
    df_mc["ts"]
    .duplicated()
    .sum()
)


print("\nUnique values per sensor:")

for column in [
    "mc_1",
    "mc_2",
    "mc_3",
    "mc_4"
]:
    print(
        column,
        sorted(
            df_mc[column].unique()
        )
    )


print("\nNumber of open windows:")

print(
    df_mc["open_windows"]
    .value_counts()
    .sort_index()
)


print("\nBinary open-window indicator:")

print(
    df_mc[
        "open_windows_binary"
    ]
    .value_counts()
)


# Convert discrete sensor states and window counts to integers.

for column in [
    "mc_1",
    "mc_2",
    "mc_3",
    "mc_4"
]:
    df_mc[column] = (
        df_mc[column]
        .astype(int)
    )

df_mc["open_windows"] = (
    df_mc["open_windows"]
    .astype(int)
)


# Export the cleaned Magnetic Contact dataset.

output_file = (
    "magnetic_contacts_4_9_clean.csv"
)

df_mc.to_csv(
    output_file,
    index=False
)


print(
    f"\nSaved: {output_file}"
)

print(
    "Final shape:",
    df_mc.shape
)

Final Magnetic Contact shape: (44277, 7)

Time range:
2025-09-01 08:30:00 → 2026-07-05 19:50:00

Missing values:
ts                     0
mc_1                   0
mc_2                   0
mc_3                   0
mc_4                   0
open_windows           0
open_windows_binary    0
dtype: int64

Duplicate timestamps:
0

Unique values per sensor:
mc_1 [np.float64(0.0), np.float64(1.0)]
mc_2 [np.float64(0.0), np.float64(1.0)]
mc_3 [np.float64(0.0), np.float64(1.0)]
mc_4 [np.float64(0.0), np.float64(1.0)]

Number of open windows:
open_windows
0.0      149
1.0    17953
2.0    13391
3.0    10002
4.0     2782
Name: count, dtype: int64

Binary open-window indicator:
open_windows_binary
1    44128
0      149
Name: count, dtype: int64

Saved: magnetic_contacts_4_9_clean.csv
Final shape: (44277, 7)


## 4. Sensor Data Integration

After preprocessing the three sensor sources independently, the resulting datasets are integrated into a single time-aligned dataset for Room 4.9.

The integration combines:

- the processed Indoor Air Quality (IAQ) measurements,
- the processed Desk Occupancy measurements,
- the processed Magnetic Contact measurements.

Since all datasets have been aligned to 10-minute intervals, they are merged using their common timestamp. An inner join is applied so that the final dataset contains only time intervals for which measurements from all three sensor sources are available.

The resulting integrated dataset is used as the main input for the exploratory data analysis and subsequent modeling stages.

The final dataset is exported as:

`room_4_9_full_dataset.csv`

In [55]:
# SAVE PREPROCESSED DATASETS FOR FINAL INTEGRATION


# Save the processed IAQ dataset generated in Section 1.
df_iaq_4_9_final.to_csv(
    "/content/iaq_4_9_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save the processed Desk Occupancy dataset generated in Section 2.
df_desks.to_csv(
    "/content/desk_occupancy_4_9_clean.csv",
    index=False
)

# Save the processed Magnetic Contact dataset generated in Section 3.
df_mc.to_csv(
    "/content/magnetic_contacts_4_9_clean.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [56]:
# 4. SENSOR DATA INTEGRATION — ROOM 4.9

#
# The three independently preprocessed sensor datasets are
# loaded and subsequently integrated using their common
# 10-minute timestamp.



iaq = pd.read_csv(
    "/content/iaq_4_9_clean.csv",
    parse_dates=["ts"]
)

desk = pd.read_csv(
    "/content/desk_occupancy_4_9_clean.csv",
    parse_dates=["ts"]
)

mc = pd.read_csv(
    "/content/magnetic_contacts_4_9_clean.csv",
    parse_dates=["ts"]
)


print("IAQ dataset:", iaq.shape)
print("Desk Occupancy dataset:", desk.shape)
print("Magnetic Contact dataset:", mc.shape)

IAQ dataset: (38834, 17)
Desk Occupancy dataset: (44067, 37)
Magnetic Contact dataset: (44277, 7)


In [57]:
# 4.1 Validate datasets before integration


datasets = {
    "IAQ": iaq,
    "Desk Occupancy": desk,
    "Magnetic Contacts": mc
}


for name, df in datasets.items():

    print(f"\n{name}")

    print(
        "Duplicate timestamps:",
        df["ts"].duplicated().sum()
    )

    print(
        "Start:",
        df["ts"].min()
    )

    print(
        "End:",
        df["ts"].max()
    )


IAQ
Duplicate timestamps: 0
Start: 2025-08-31 21:00:00
End: 2026-06-05 06:50:00

Desk Occupancy
Duplicate timestamps: 0
Start: 2025-09-01 20:40:00
End: 2026-07-04 21:00:00

Magnetic Contacts
Duplicate timestamps: 0
Start: 2025-09-01 08:30:00
End: 2026-07-05 19:50:00


In [58]:
# 4.2 Integrate the three sensor datasets

# Inner joins are used to retain only timestamps that are
# simultaneously available in all three processed datasets.


room_4_9 = (
    iaq
    .merge(
        desk,
        on="ts",
        how="inner"
    )
    .merge(
        mc,
        on="ts",
        how="inner"
    )
)


# Sort the final dataset chronologically.

room_4_9 = (
    room_4_9
    .sort_values("ts")
    .reset_index(drop=True)
)


print(
    "Integrated dataset shape:",
    room_4_9.shape
)

display(
    room_4_9.head()
)

display(
    room_4_9.tail()
)

Integrated dataset shape: (38692, 59)


,ts,co2_iaq1,temperature_iaq1,humidity_iaq1,light_level_iaq1,pir_iaq1,co2_iaq2,temperature_iaq2,humidity_iaq2,light_level_iaq2,...,occupied_desks,free_desks,occupancy_rate,any_desk_occupied,mc_1,mc_2,mc_3,mc_4,open_windows,open_windows_binary
0,2025-09-01 20:40:00,583.0,29.3,42.5,0.0,0.0,577.0,29.2,42.5,0.0,...,0,32,0.0,0,1,0,0,0,1,1
1,2025-09-01 20:50:00,580.0,29.3,42.5,0.0,0.0,575.0,29.2,42.5,0.0,...,0,32,0.0,0,1,0,0,0,1,1
2,2025-09-01 21:00:00,577.0,29.3,42.5,0.0,0.0,571.0,29.2,42.5,0.0,...,0,32,0.0,0,1,0,0,0,1,1
3,2025-09-01 21:10:00,572.0,29.3,42.5,0.0,0.0,566.0,29.2,43.0,0.0,...,0,32,0.0,0,1,0,0,0,1,1
4,2025-09-01 21:20:00,568.0,29.3,42.5,0.0,0.0,561.0,29.2,43.0,0.0,...,0,32,0.0,0,1,0,0,0,1,1


,ts,co2_iaq1,temperature_iaq1,humidity_iaq1,light_level_iaq1,pir_iaq1,co2_iaq2,temperature_iaq2,humidity_iaq2,light_level_iaq2,...,occupied_desks,free_desks,occupancy_rate,any_desk_occupied,mc_1,mc_2,mc_3,mc_4,open_windows,open_windows_binary
38687,2026-05-28 12:30:00,505.848079,26.100537,42.504472,0.0,1.0,NaN,NaN,NaN,NaN,...,1,31,0.03125,1,1,1,1,0,3,1
38688,2026-05-28 12:40:00,505.772118,26.100805,42.506708,0.0,1.0,NaN,NaN,NaN,NaN,...,1,31,0.03125,1,1,1,1,0,3,1
38689,2026-06-05 06:30:00,421.075961,26.400000,45.000000,3.0,1.0,NaN,NaN,NaN,NaN,...,0,32,0.00000,0,1,1,1,0,3,1
38690,2026-06-05 06:40:00,421.000000,26.400000,44.500000,3.0,1.0,NaN,NaN,NaN,NaN,...,0,32,0.00000,0,1,1,1,0,3,1
38691,2026-06-05 06:50:00,424.000000,26.500000,44.500000,3.0,0.0,NaN,NaN,NaN,NaN,...,0,32,0.00000,0,1,1,1,0,3,1


In [59]:
# 4.3 Final integrated dataset validation


print(
    "Final dataset shape:",
    room_4_9.shape
)


print("\nTime range:")

print(
    room_4_9["ts"].min(),
    "→",
    room_4_9["ts"].max()
)


print("\nDuplicate timestamps:")

print(
    room_4_9["ts"]
    .duplicated()
    .sum()
)


print("\nMissing values:")

print(
    room_4_9.isna().sum()
)


print("\nAvailable IAQ sensors:")

print(
    room_4_9[
        "available_iaq_sensors"
    ]
    .value_counts()
    .sort_index()
)


print("\nOccupied desks statistics:")

print(
    room_4_9[
        "occupied_desks"
    ]
    .describe()
)


print("\nOpen windows statistics:")

print(
    room_4_9[
        "open_windows"
    ]
    .describe()
)

Final dataset shape: (38692, 59)

Time range:
2025-09-01 20:40:00 → 2026-06-05 06:50:00

Duplicate timestamps:
0

Missing values:
ts                          0
co2_iaq1                   12
temperature_iaq1           12
humidity_iaq1              12
light_level_iaq1            0
pir_iaq1                    0
co2_iaq2                 1765
temperature_iaq2         1765
humidity_iaq2            1765
light_level_iaq2         1765
pir_iaq2                 1765
co2_avg                     0
temperature_avg             0
humidity_avg                0
light_level_avg             0
pir_any                     0
available_iaq_sensors       0
desk_1                      0
desk_2                      0
desk_3                      0
desk_4                      0
desk_5                      0
desk_6                      0
desk_7                      0
desk_8                      0
desk_9                      0
desk_10                     0
desk_11                     0
desk_12                     0


In [60]:
# 4.4 Data types and final export



# Convert count variables to integer representation.

integer_columns = [
    "occupied_desks",
    "free_desks",
    "open_windows"
]


for column in integer_columns:

    room_4_9[column] = (
        room_4_9[column]
        .astype(int)
    )


# Export the integrated Room 4.9 dataset.

output_file = (
    "room_4_9_full_dataset.csv"
)

room_4_9.to_csv(
    output_file,
    index=False
)


print(
    f"Saved: {output_file}"
)

print(
    "Final shape:",
    room_4_9.shape
)

Saved: room_4_9_full_dataset.csv
Final shape: (38692, 59)
